# Learning 3: Simple Chains

**Goal**: Connect components together using the LCEL pipe operator

## What You'll Learn
- What chains are and why they're useful
- Using the `|` pipe operator
- Chaining prompts, LLMs, and output parsers
- Creating multi-step chains

In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini")
print("Setup complete!")

/Users/syedraza/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Setup complete!


## What is a Chain?

A chain connects multiple components:

```
[Input] → [Prompt] → [LLM] → [Output Parser] → [Result]
```

Instead of calling each component separately, chains let you combine them into a single callable.

## The Pipe Operator `|`

LangChain uses the pipe operator `|` to chain components. It's called LCEL (LangChain Expression Language).

In [6]:
# Without chaining (verbose) - Manual step-by-step
prompt = ChatPromptTemplate.from_template("Translate '{text}' to {language}")
formatted = prompt.format_messages(text="Hello, how are you?", language="Spanish")
response = llm.invoke(formatted)
text = response.content

print("Without chain (manual steps):", text)

Without chain (manual steps): 'Hello, how are you?' in Spanish is 'Hola, ¿cómo estás?'


In [7]:
# With chaining (clean!) - All in one line
prompt = ChatPromptTemplate.from_template("Translate '{text}' to {language}")
output_parser = StrOutputParser()

chain = prompt | llm | output_parser

result = chain.invoke({"text": "Hello, how are you?", "language": "French"})
print("With chain (one call):", result)

With chain (one call): The translation of "Hello, how are you?" to French is "Bonjour, comment ça va ?"


## Understanding StrOutputParser

The `StrOutputParser` extracts just the text content from the LLM response.

In [9]:
# Without parser - returns AIMessage object
fact_prompt = ChatPromptTemplate.from_template("Tell me a fact about {topic}")
chain_no_parser = fact_prompt | llm
result = chain_no_parser.invoke({"topic": "birds"})
print("Type:", type(result))
print("Result:", result)

Type: <class 'langchain_core.messages.ai.AIMessage'>
Result: content='Birds are the only group of animals that have feathers, which are essential for flight, insulation, and display. Feathers are unique to birds and are thought to have evolved from the scales of their reptilian ancestors.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 13, 'total_tokens': 57, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c4585b5b9c', 'id': 'chatcmpl-D1mOAbeCmXcYVDxNQ0LSXKy0qaaWU', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--29fd6f44-4b73-4d4c-aa8f-a1ddd8581342-0' usage_metadata={'input_tokens': 13, 'output_tokens': 44, 'total_tokens': 57, 'input_token_details': {'audio': 0, 'cache_

In [11]:
# With parser - returns just the string
chain_with_parser = fact_prompt | llm | StrOutputParser()
result = chain_with_parser.invoke({"topic": "birds"})
print("Type:", type(result))
print("Result:", result)

Type: <class 'str'>
Result: Birds are the only group of animals that have feathers, which are unique to them. Feathers play a crucial role in flight, insulation, and display in various species, making them key to the success of birds in diverse environments.


## Multi-Step Chain

Chain the output of one LLM call as input to another.

In [12]:
# Step 1: Extract key points from text
extract_prompt = ChatPromptTemplate.from_template(
    "Extract 3 key points from this text in a bullet list:\n\n{text}"
)

# Step 2: Summarize those points
summarize_prompt = ChatPromptTemplate.from_template(
    "Write a one-sentence summary based on these points:\n\n{key_points}"
)

# Chain step 1
extract_chain = extract_prompt | llm | StrOutputParser()

# Test step 1
sample_text = "Artificial intelligence is transforming industries. Machine learning enables computers to learn from data. Deep learning uses neural networks to solve complex problems."
key_points = extract_chain.invoke({"text": sample_text})
print("Key points extracted:")
print(key_points)

Key points extracted:
- Artificial intelligence is significantly transforming various industries.
- Machine learning allows computers to learn and adapt from data.
- Deep learning employs neural networks to address complex problem-solving tasks.


In [13]:
# Combine both chains - extract then summarize
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Full pipeline: text → key points → summary
full_chain = (
    {"key_points": extract_prompt | llm | StrOutputParser()}
    | summarize_prompt
    | llm
    | StrOutputParser()
)

new_text = "Cloud computing provides on-demand access to computing resources. It offers scalability and cost efficiency. Major providers include AWS, Azure, and Google Cloud."
summary = full_chain.invoke({"text": new_text})
print("Summary:", summary)

Summary: Cloud computing delivers on-demand access to scalable and cost-efficient computing resources, with major providers including AWS, Azure, and Google Cloud.


## Parallel Chains with RunnableParallel

Run multiple chains at once and combine their outputs.

In [14]:
from langchain_core.runnables import RunnableParallel

# Create parallel chains
joke_prompt = ChatPromptTemplate.from_template("Tell a short joke about {topic}")
fact_prompt = ChatPromptTemplate.from_template("Tell a fact about {topic}")

parallel_chain = RunnableParallel(
    joke=joke_prompt | llm | StrOutputParser(),
    fact=fact_prompt | llm | StrOutputParser()
)

result = parallel_chain.invoke({"topic": "coffee"})
print("Joke:", result["joke"])
print("\nFact:", result["fact"])

Joke: Why do coffee beans never get in trouble?  

Because they know how to espresso themselves!

Fact: One interesting fact about coffee is that it is actually a fruit! The coffee beans we use to make the beverage are the seeds of the coffee cherry, which is a small, red or purple fruit that grows on coffee plants. Each cherry typically contains two seeds (the coffee beans) inside.


## Adding Custom Functions with RunnableLambda

In [15]:
from langchain_core.runnables import RunnableLambda

# Custom function to process output
def make_uppercase(text: str) -> str:
    return text.upper()

def add_emoji(text: str) -> str:
    return f"✨ {text} ✨"

# Chain with custom functions
chain = (
    ChatPromptTemplate.from_template("Say hello to {name}")
    | llm
    | StrOutputParser()
    | RunnableLambda(make_uppercase)
    | RunnableLambda(add_emoji)
)

result = chain.invoke({"name": "Alice"})
print(result)

✨ HELLO, ALICE! HOW ARE YOU TODAY? ✨


## Streaming with Chains

Chains support streaming too!

In [ ]:
chain = (
    ChatPromptTemplate.from_template("Write a short product description for {product}")
    | llm
    | StrOutputParser()
)

print("Streaming product description:")
for chunk in chain.stream({"product": "a smart water bottle that tracks hydration"}):
    print(chunk, end="", flush=True)
print("\n")

Streaming:
Lines ofLines of code align code align,  
,  
Logic dancesLogic dances through the through the night, night,  
Dream  
Dreams ins in bytes take bytes take flight. flight.

## Real-World Example: Content Moderator + Rewriter

Let's build a practical chain that:
1. Analyzes text for tone/issues
2. Decides if it needs rewriting
3. Rewrites if needed

In [ ]:
from pydantic import BaseModel, Field

# Step 1: Analyze the text
class TextAnalysis(BaseModel):
    is_appropriate: bool = Field(description="Whether the text is professional and appropriate")
    reason: str = Field(description="Brief reason for the assessment")

analyze_prompt = ChatPromptTemplate.from_template(
    "Analyze if this text is professional and appropriate for a business email:\n\n{text}\n\n"
    "Consider tone, language, and professionalism."
)

analyze_chain = analyze_prompt | llm.with_structured_output(TextAnalysis)

# Test the analyzer
sample_text = "Hey dude, your idea is kinda stupid tbh. We should do something else."
analysis = analyze_chain.invoke({"text": sample_text})
print(f"Appropriate: {analysis.is_appropriate}")
print(f"Reason: {analysis.reason}")

In [ ]:
# Step 2: Rewrite inappropriate text
rewrite_prompt = ChatPromptTemplate.from_template(
    "Rewrite this text to be professional and appropriate for a business email:\n\n{text}"
)

rewrite_chain = rewrite_prompt | llm | StrOutputParser()

# Test the rewriter
rewritten = rewrite_chain.invoke({"text": sample_text})
print("Original:", sample_text)
print("\nRewritten:", rewritten)

NameError: name 'sample_text' is not defined

In [ ]:
# Step 3: Combine both into a smart chain with conditional logic
def smart_moderate(input_dict):
    """Analyzes text and only rewrites if inappropriate"""
    text = input_dict["text"]
    
    # Analyze
    analysis = analyze_chain.invoke({"text": text})
    
    # If appropriate, return original; otherwise rewrite
    if analysis.is_appropriate:
        return {
            "original": text,
            "result": text,
            "was_rewritten": False,
            "reason": analysis.reason
        }
    else:
        rewritten = rewrite_chain.invoke({"text": text})
        return {
            "original": text,
            "result": rewritten,
            "was_rewritten": True,
            "reason": analysis.reason
        }

# Wrap in RunnableLambda to make it chainable
moderator_chain = RunnableLambda(smart_moderate)

# Test with inappropriate text
print("Test 1 - Inappropriate:")
result1 = moderator_chain.invoke({"text": "Hey dude, your idea is kinda stupid tbh."})
print(f"Rewritten: {result1['was_rewritten']}")
print(f"Reason: {result1['reason']}")
print(f"Result: {result1['result']}")

print("\n" + "="*60 + "\n")

# Test with appropriate text
print("Test 2 - Appropriate:")
result2 = moderator_chain.invoke({"text": "Thank you for your proposal. I have some suggestions for improvement."})
print(f"Rewritten: {result2['was_rewritten']}")
print(f"Reason: {result2['reason']}")
print(f"Result: {result2['result']}")

## Exercise: Build Your Own Chain

1. Create a chain that translates text and then summarizes it
2. Create parallel chains for different types of analysis
3. Add a custom function to your chain

In [ ]:
# Your code here!



## Key Takeaways

1. Use `|` to chain components together
2. `StrOutputParser()` extracts text from LLM responses
3. `RunnableParallel` runs multiple chains simultaneously
4. `RunnableLambda` wraps custom functions for use in chains
5. Chains support `.invoke()`, `.stream()`, and `.batch()`

**Next**: Learning 4 - Tools Basics